In [ ]:
from itertools import product
import nest_asyncio
import asyncio
from google import genai
import os
import json
import pandas as pd
from dotenv import load_dotenv

nest_asyncio.apply()
_llm_instance = None

load_dotenv()

GENAI_API_KEY = os.getenv("GENAI_API_KEY")
if not GENAI_API_KEY:
    print(" Warning: GENAI_API_KEY not set in environment. Please set it in .env or the environment.")
GEMINI_MODEL = os.getenv("GEMINI_MODEL")
client = genai.Client(api_key=GENAI_API_KEY)


async def call_llm(prompt):
    """
    Interface with the LLM API to process the given prompt.
    Uses Google's Gemini model.
    """
    try:
        print("→ Sending prompt to Gemini...")

        # Run the blocking call in a thread
        response = await asyncio.to_thread(
            client.models.generate_content,
            model=GEMINI_MODEL,
            contents=prompt
        )

        #  Gemini returns text in .text (not .message.content)
        if hasattr(response, "text") and response.text:
            return response.text.strip()

        # Fallback: Try nested structure (depends on SDK version)
        if hasattr(response, "candidates") and response.candidates:
            parts = response.candidates[0].content.parts
            if parts and hasattr(parts[0], "text"):
                return parts[0].text.strip()

        # If still nothing, return placeholder  
        return "LLM returned an empty response."

    except Exception as e:
        print(f"Error calling LLM: {e}")
        return ""



BASE_PROMPT = """
You are an expert in Indian packaged food and beverage products.

For each product below:
1. First, internally reason whether the product is a legitimate SKU sold in India.
2. Consider brand credibility, common variants, and known market presence.
3. Be conservative: only flag when you are reasonably confident the product is invalid, discontinued, or a duplicate with identical ingredients and nutrition by verifying with the market online.

FINAL OUTPUT RULES (STRICT):
- Output EXACTLY one word per line
- Allowed outputs ONLY:
  Flag
  Let it be
- Do NOT explain
- Do NOT add headers
- Do NOT number items
- Keep the same order as input

Products:
{products}
"""




async def classify_dataframe_in_batches(
    df: pd.DataFrame,
    batch_size: int = 10,
    sleep_seconds: int = 25
) -> pd.DataFrame:

    all_flags = []

    for start in range(0, len(df), batch_size):
        batch = df.iloc[start:start + batch_size]

        products_text = "\n".join(
            f"{row['brand']} | {row['sku_name']}"
            for _, row in batch.iterrows()
        )

        prompt = BASE_PROMPT.format(products=products_text)

        print(f"🔹 Processing batch {start} → {start + len(batch)}")

        response = await call_llm(prompt)
        response = (response or "").strip()

        # Parse output safely
        lines = [line.strip() for line in response.splitlines() if line.strip()]

        # Fallback if output is malformed
        if len(lines) != len(batch):
            print("Output mismatch, defaulting batch to Flag")
            batch_flags = ["Flag"] * len(batch)
        else:
            batch_flags = [
                line if line in {"Flag", "Let it be"} else "Flag"
                for line in lines
            ]

        all_flags.extend(batch_flags)
        print("Batch results:")
        for p, f in zip(batch["brand"] + " " + batch["sku_name"], batch_flags):
            print(f"  {p} → {f}")

        # CRITICAL for free tier
        if start + batch_size < len(df):
            print(f" Sleeping {sleep_seconds}s to respect rate limits...")
            await asyncio.sleep(sleep_seconds)

    df["Product"] = df["brand"].astype(str) + " " + df["sku_name"].astype(str)
    df["Flag"] = all_flags

    return df[["Product", "Flag"]]


    

async def main():
    # Read Excel file
    df = pd.read_excel("Food_investigation.xlsx")

    # Validate required columns
    required_cols = {"brand", "sku_name"}
    if not required_cols.issubset(df.columns):
        raise ValueError("Excel must contain 'brand' and 'sku_name' columns")

    # Run async pipeline
    result_df = await classify_dataframe_in_batches(df, batch_size=10)

    # Save result
    print(f"\nresults:\n{result_df}")
    result_df.to_excel("results.xlsx", index=False)


In [3]:
asyncio.run(main())

🔹 Processing batch 0 → 10
→ Sending prompt to Gemini...
✅ Batch results:
  healthy & tasty emami healthy tasty refined sunflower oil → Let it be
  healthy & tasty emami healthy tasty refined soybean oil → Let it be
  healthy & tasty emami healthy tasty refined rice bran oil → Let it be
  healthy & tasty emami healthy tasty blended oil → Let it be
  healthy & tasty emami healthy tasty fresh chakki atta → Let it be
  healthy & tasty emami healthy tasty maida → Let it be
  healthy & tasty emami healthy tasty suji → Let it be
  healthy & tasty emami healthy tasty advans soya chunks → Let it be
  mantra mantra pure turmeric powder → Let it be
  mantra mantra pure chilli powder → Let it be
⏳ Sleeping 25s to respect rate limits...
🔹 Processing batch 10 → 20
→ Sending prompt to Gemini...
✅ Batch results:
  mantra mantra pure cumin powder → Let it be
  mantra mantra pure coriander powder → Let it be
  mantra mantra garam masala blend → Let it be
  mantra mantra meat masala → Let it be
  mantra 